# 人工框評估與 AP50

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/06-evaluation/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Image-aware, class-aware, one-use GT matching and all-points interpolated AP."""
import torch


def iou(box, boxes):
    intersection = (torch.minimum(box[2:], boxes[:, 2:]) -
                    torch.maximum(box[:2], boxes[:, :2])).clamp(min=0).prod(1)
    area = (box[2:] - box[:2]).clamp(min=0).prod()
    areas = (boxes[:, 2:] - boxes[:, :2]).clamp(min=0).prod(1)
    return intersection / (area + areas - intersection).clamp(min=1e-8)


def interpolated_ap(recall, precision):
    r = torch.cat((torch.tensor([0.]), recall, torch.tensor([1.])))
    p = torch.cat((torch.tensor([0.]), precision, torch.tensor([0.])))
    for i in range(len(p) - 2, -1, -1):
        p[i] = torch.maximum(p[i], p[i + 1])
    changes = (r[1:] != r[:-1]).nonzero().flatten()
    return ((r[changes + 1] - r[changes]) * p[changes + 1]).sum().item()


def evaluate(predictions, targets, num_classes=2, match_iou_threshold=.5):
    aps, records = [], {}
    total_tp = total_fp = total_gt = 0
    for cls in range(num_classes):
        gt_count = sum((t["labels"] == cls).sum().item() for t in targets)
        candidates = [(p["scores"][j].item(), b, j)
                      for b, p in enumerate(predictions)
                      for j in range(len(p["boxes"])) if p["labels"][j] == cls]
        candidates.sort(key=lambda entry: (-entry[0], entry[1], entry[2]))
        matched = [set() for _ in targets]
        flags = []
        for _, b, j in candidates:
            eligible = [k for k in range(len(targets[b]["boxes"]))
                        if targets[b]["labels"][k] == cls and k not in matched[b]]
            is_tp = False
            if eligible:
                overlaps = iou(predictions[b]["boxes"][j], targets[b]["boxes"][eligible])
                best = overlaps.argmax().item()
                if overlaps[best] >= match_iou_threshold:
                    matched[b].add(eligible[best])
                    is_tp = True
            flags.append(int(is_tp))
        flags_tensor = torch.tensor(flags, dtype=torch.float32)
        tp = flags_tensor.cumsum(0)
        precision = tp / torch.arange(1, len(flags) + 1)
        recall = tp / max(gt_count, 1)
        ap = interpolated_ap(recall, precision) if gt_count else None
        aps.append(ap)
        records[cls] = {"flags": flags, "precision": precision, "recall": recall,
                        "scores": [entry[0] for entry in candidates]}
        total_tp += int(flags_tensor.sum().item())
        total_fp += len(flags) - int(flags_tensor.sum().item())
        total_gt += gt_count
    valid_aps = [ap for ap in aps if ap is not None]
    return {"ap_per_class": aps, "map": sum(valid_aps) / len(valid_aps) if valid_aps else None,
            "precision": total_tp / max(total_tp + total_fp, 1),
            "recall": total_tp / max(total_gt, 1),
            "tp": total_tp, "fp": total_fp, "fn": total_gt - total_tp, "records": records}


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    targets = [
        {"boxes": torch.tensor([[0., 0., 10., 10.], [20., 0., 30., 10.]]), "labels": torch.tensor([0, 0])},
        {"boxes": torch.tensor([[0., 20., 10., 30.]]), "labels": torch.tensor([0])},
    ]
    predictions = [
        {"boxes": torch.tensor([[40., 40., 50., 50.], [0., 0., 10., 10.], [1., 0., 11., 10.]]),
         "scores": torch.tensor([.95, .9, .8]), "labels": torch.tensor([0, 0, 0])},
        {"boxes": torch.tensor([[0., 20., 10., 30.]]), "scores": torch.tensor([.7]), "labels": torch.tensor([0])},
    ]
    result = evaluate(predictions, targets)
    record = result["records"][0]
    assert record["flags"] == [0, 1, 0, 1]
    assert (result["tp"], result["fp"], result["fn"]) == (2, 2, 1)
    assert abs(result["precision"] - .5) < 1e-6 and abs(result["recall"] - 2 / 3) < 1e-6
    assert abs(result["map"] - 1 / 3) < 1e-6 and result["ap_per_class"][1] is None
    for rank, (score, flag, p, r) in enumerate(zip(record["scores"], record["flags"],
                                                 record["precision"], record["recall"]), 1):
        print(f"rank={rank}, score={score:.2f}, {'TP' if flag else 'FP'}, "
              f"precision={p.item():.4f}, recall={r.item():.4f}")
    # mAP50 averages AP only over classes that have GT; class 1 has none, so its AP is None.
    ap_per_class = [None if ap is None else round(ap, 6) for ap in result["ap_per_class"]]
    print(f"TP={result['tp']}, FP={result['fp']}, FN={result['fn']}, "
          f"mAP50={result['map']:.6f}, ap_per_class={ap_per_class}")
    truncated = [{key: values[p["scores"] >= .85] for key, values in p.items()} for p in predictions]
    short_result = evaluate(truncated, targets)
    assert abs(short_result["map"] - 1 / 6) < 1e-6
    print(f"candidate threshold .85: mAP50={short_result['map']:.6f}, recall={short_result['recall']:.4f}")
    # Remove a known artificial false positive, without changing GT.
    cleaned = [{"boxes": predictions[0]["boxes"][1:], "scores": predictions[0]["scores"][1:],
                "labels": predictions[0]["labels"][1:]}, predictions[1]]
    clean_result = evaluate(cleaned, targets)
    assert abs(clean_result["map"] - 5 / 9) < 1e-6
    print(f"remove known high-score FP: mAP50={clean_result['map']:.6f}")
    empty = {"boxes": torch.empty(0, 4), "scores": torch.empty(0), "labels": torch.empty(0, dtype=torch.long)}
    no_predictions = evaluate([empty, empty], targets)
    assert no_predictions["map"] == 0 and no_predictions["fn"] == 3
    # An empty image can still contribute false positives.
    empty_target = {"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)}
    background = evaluate([predictions[1]], [empty_target])
    assert background["fp"] == 1 and background["map"] is None
    print("no predictions => AP=0 when GT exists; no-GT class AP=None; background FP counted")
    print("Artificial scoring exercise; not measured detector performance.")


if __name__ == "__main__":
    main()


rank=1, score=0.95, FP, precision=0.0000, recall=0.0000
rank=2, score=0.90, TP, precision=0.5000, recall=0.3333
rank=3, score=0.80, FP, precision=0.3333, recall=0.3333
rank=4, score=0.70, TP, precision=0.5000, recall=0.6667
TP=2, FP=2, FN=1, mAP50=0.333333, ap_per_class=[0.333333, None]
candidate threshold .85: mAP50=0.166667, recall=0.3333
remove known high-score FP: mAP50=0.555556
no predictions => AP=0 when GT exists; no-GT class AP=None; background FP counted
Artificial scoring exercise; not measured detector performance.
